# The REG402_EVENT_TEXT table 

Welcome to the detailed explanation of the `REG402_EVENT_TEXT` reference table. This table belongs to category 400 of the PATSTAT EP Register tables. Tables in this category are used as code lists, meaning they provide standardised codes and explanations that are referenced by other tables in the database.

The `REG402_EVENT_TEXT` table specifically contains event text codes. These codes are used in the `REG301_EVENT_DATA` table. For each code, `REG402_EVENT_TEXT` provides a clear, easy-to-read description that explains what the event means. A “Legal Event” refers to a procedural action that changes the legal status of a patent application or a granted patent. In simple terms, these events record important steps or decisions that affect how a patent application or patent progresses over time.

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG301_EVENT_DATA, REG402_EVENT_TEXT
from sqlalchemy import select, func, case, select, and_

patstat = PatstatClient(env='TEST')

db = patstat.orm()

## Key Fields in the REG402_EVENT_TEXT Table

### EVENT_CODE (primary key)
This field contains the internal code used by the EPO to identify a specific type of action, such as publications, changes or deletions, which occurs throughout the patent process. It serves as a technical identifier, allowing systems to categorize the exact nature of the procedural event taking place. The code is up to 30 characters long and may include descriptive elements, such as "9199" for actions before B1 publication or "9299" for actions after B1 publication. It helps ensure consistent identification and understanding of each specific event. A list of all legal events can be viewed or downloaded as XML file from the [EPO website](https://register.epo.org/allEvents?lng=en).

### EVENT_TEXT
This field contains the description of the action in clear text. It provides the human-readable explanation corresponding to the `EVENT_CODE`, clarifying exactly what action or event occurred.

In [2]:
q = (
    db.query(
        REG402_EVENT_TEXT.event_code,
        REG402_EVENT_TEXT.event_text
    )
    .order_by(REG402_EVENT_TEXT.event_code)
    .distinct()
)

res = patstat.df(q)
res

,event_code,event_text
0,0008199APPR,Change - applicant
1,0008199DANR,Change - divisional application(s)
2,0008199FREP,Change - representative
3,0008199INVT,Change - inventor
4,0008199IPCL,Change - classification
...,...,...
318,EPIDOSNRVR2,New entry: Kind of request for revocation
319,EPIDOSNSFEE,New entry: PCT data prior to European publicat...
320,EPIDOSNTIPA,New entry: Observations by third parties
321,EPIDOSNVAP2,New entry: Fee payment for validation state


## Linking REG301_EVENT_DATA to REG402_EVENT_TEXT
To enrich the information contained in `REG301_EVENT_DATA`, we can join it with the `REG402_EVENT_TEXT` reference table using the `EVENT_CODE` attribute. This join replaces an internal EPO technical code with meaningful contextual information, namely the clear text description (`EVENT_TEXT`) of the legal or procedural action.

In [3]:
q = (
    db.query(
        REG301_EVENT_DATA.id,
        REG301_EVENT_DATA.event_date,
        REG301_EVENT_DATA.event_code,
        REG402_EVENT_TEXT.event_text,
    )
    .join(
        REG402_EVENT_TEXT,
        REG402_EVENT_TEXT.event_code == REG301_EVENT_DATA.event_code
    )
    .order_by(REG301_EVENT_DATA.id)
    .distinct()
    .limit(1000)
)

res = patstat.df(q)
res

,id,event_date,event_code,event_text
0,100008,2002-02-19,EPIDOS RFEE,Renewal fee
1,100008,2000-06-09,0009012,Publication in section I.1 EP Bulletin
2,100008,2004-02-05,EPIDOSNRFE2,New entry: Renewal fee paid
3,100008,2003-02-07,EPIDOS RFEE,Renewal fee
4,100008,2002-05-31,0009199APPR,Change - applicant
...,...,...,...,...
995,410139,2003-11-11,EPIDOSNRFE2,New entry: Renewal fee paid
996,410139,2004-11-19,EPIDOSNRFE2,New entry: Renewal fee paid
997,410139,2005-11-18,EPIDOSNRFE2,New entry: Renewal fee paid
998,410139,2006-11-18,EPIDOSNRFE2,New entry: Renewal fee paid
